In [0]:
%run ../01_Bronze/00_setup

In [0]:
# process new records into the Silver layer order_items table
from pyspark.sql import functions as F

def process_scd1_order_items(microBatchDF, batchId):
    # 1. Get a list of order_ids that already exist in Silver
    # This prevents duplicating brand new orders
    target_table = "workspace.amazon_fullfilment_silver.order_items_silver"
    
    # 2. Split the micro-batch logic
    # Part A: The 'Insert' half (Always NULL merge_key)
    inserts_df = microBatchDF.withColumn("merge_key", F.lit(None))
    
    # Part B: The 'Update' half (Only for IDs that already exist in Silver)
    # This prevents the 'double insert' on the first run
    existing_ids = microBatchDF.sparkSession.table(target_table).select("order_item_id")
       # .filter("is_current = true") \
    
    
    updates_df = microBatchDF.join(existing_ids, "order_item_id", "inner") \
        .withColumn("merge_key", F.col("order_item_id"))
    
    # 3. Combine and Merge
    combined_df = inserts_df.unionByName(updates_df, allowMissingColumns=True)
    #display(combined_df)
    combined_df.createOrReplaceTempView("source_order_items_view")

    microBatchDF.sparkSession.sql(f"""
        MERGE INTO {target_table} AS target
        USING source_order_items_view AS source
        ON target.order_item_id = source.merge_key
        
        -- If we find the ID and status changed, expire the old one
        WHEN MATCHED AND source.product_id <> target.product_id AND source.quantity <> target.quantity THEN
            UPDATE SET 
            target.product_id = source.product_id,
            target.quantity = source.quantity,
            target.updated_timestamp = current_timestamp()
          
        -- If it's the 'NULL' half of the union, it will always be NOT MATCHED -> Insert
        WHEN NOT MATCHED THEN
          INSERT (
            order_item_sk, 
            order_item_id, 
            order_id, 
            product_id, 
            quantity, 
            unit_price, 
            updated_timestamp
          )
          VALUES (
            md5(source.order_item_id), 
            source.order_item_id, 
            source.order_id, 
            source.product_id, 
            source.quantity, 
            0.00, 
            current_timestamp()
          )
    """)

    # 3. Start the Stream
query = (spark.readStream
    .table("workspace.amazon_fullfilment_bronze.order_items")
    .filter("order_item_id IS NOT NULL") # Basic data quality
    .writeStream
    .foreachBatch(process_scd1_order_items)
    .option("checkpointLocation", f"{silver_volume_path}/checkpoints/silver_order_items_scd1")
    .trigger(availableNow=True) # Or .trigger(processingTime='1 minute') for 24/7
    .start())